# Multi-Agent Emergency Response System
### Agent Dispatch Module - Manya Mahajan (AI Agent Developer)
**Sprint 1 | Week 2–3**

This notebook builds the multi-agent dispatch system for the emergency response project.
It covers:
- Emergency type definitions and dispatch rules
- Base agent class
- Ambulance, Fire, and Police agents
- Central dispatch function
- Test scenarios


---
## Cell 1 - Emergency Types & Dispatch Table

This is the brain of the dispatch system. It defines:
1. Every emergency type the system can recognise
2. Which agents should respond to each type
3. The priority level (1 = highest)

This table is used by the dispatch function in Cell 6 to decide which agents to activate.

In [3]:
# ─────────────────────────────────────────────
# CELL 1 — Emergency Types & Dispatch Table
# ─────────────────────────────────────────────

# Each emergency type maps to:
# - 'agents'   : which agents respond (can be multiple)
# - 'priority' : 1 = critical, 2 = high, 3 = medium
# - 'notes'    : human-readable explanation

DISPATCH_TABLE = {
    "cardiac_arrest": {
        "agents": ["ambulance"],
        "priority": 1,
        "notes": "Medical emergency — ambulance only"
    },
    "fire": {
        "agents": ["fire", "ambulance"],
        "priority": 1,
        "notes": "Fire service leads, ambulance on standby for injuries"
    },
    "car_accident_minor": {
        "agents": ["ambulance", "police"],
        "priority": 2,
        "notes": "Ambulance for injuries, police for traffic control"
    },
    "car_accident_major": {
        "agents": ["ambulance", "fire", "police"],
        "priority": 1,
        "notes": "All services — entrapment likely, major injuries expected"
    },
    "robbery": {
        "agents": ["police"],
        "priority": 2,
        "notes": "Police only"
    },
    "assault": {
        "agents": ["police", "ambulance"],
        "priority": 2,
        "notes": "Police to secure scene, ambulance for victim"
    },
    "building_collapse": {
        "agents": ["fire", "ambulance", "police"],
        "priority": 1,
        "notes": "All services — search and rescue situation"
    },
    "gas_leak": {
        "agents": ["fire", "police"],
        "priority": 1,
        "notes": "Fire handles hazard, police for evacuation"
    },
    "drowning": {
        "agents": ["ambulance", "police"],
        "priority": 1,
        "notes": "Ambulance for resuscitation, police for scene control"
    },
    "unknown": {
        "agents": ["police"],
        "priority": 3,
        "notes": "Default — police assess and call in other services if needed"
    }
}

# ── Quick preview ──────────────────────────────
print(f"{'EMERGENCY TYPE':<25} {'AGENTS':<35} {'PRIORITY'}")
print("─" * 70)
for emergency, details in DISPATCH_TABLE.items():
    agents_str = ", ".join(details["agents"])
    print(f"{emergency:<25} {agents_str:<35} {details['priority']}")

EMERGENCY TYPE            AGENTS                              PRIORITY
──────────────────────────────────────────────────────────────────────
cardiac_arrest            ambulance                           1
fire                      fire, ambulance                     1
car_accident_minor        ambulance, police                   2
car_accident_major        ambulance, fire, police             1
robbery                   police                              2
assault                   police, ambulance                   2
building_collapse         fire, ambulance, police             1
gas_leak                  fire, police                        1
drowning                  ambulance, police                   1
unknown                   police                              3


---
## Cell 2 - Base Agent Class

Before building the individual agents, we create a **base `Agent` class** that all 
three agents inherit from.

This avoids repeating the same code three times and ensures every agent:
- Has a name and a type
- Tracks whether it is available or busy
- Has a standard `respond()` method that can be overridden
- Logs what it does

Think of this as the job contract every agent must follow.

In [5]:
# ─────────────────────────────────────────────
# CELL 2 — Base Agent Class
# ─────────────────────────────────────────────

from datetime import datetime

class Agent:
    """
    Base class for all emergency response agents.
    Ambulance, Fire, and Police agents all inherit from this.
    """

    def __init__(self, agent_id, agent_type, location="DEPOT"):
        self.agent_id   = agent_id        # e.g. "AMB_01"
        self.agent_type = agent_type      # "ambulance" | "fire" | "police"
        self.location   = location        # placeholder — real coords from Diyona in Sprint 2
        self.status     = "available"     # "available" | "busy"
        self.log        = []              # activity log for this agent

    def is_available(self):
        return self.status == "available"

    def assign(self, emergency):
        """
        Assign this agent to an emergency.
        Marks agent as busy and logs the assignment.
        """
        if not self.is_available():
            return f"{self.agent_id} is currently busy and cannot respond."

        self.status = "busy"
        timestamp   = datetime.now().strftime("%H:%M:%S")
        entry       = f"[{timestamp}] {self.agent_id} assigned to: {emergency}"
        self.log.append(entry)
        return entry

    def complete(self):
        """Mark the agent as available again once the job is done."""
        self.status = "available"
        timestamp   = datetime.now().strftime("%H:%M:%S")
        entry       = f"[{timestamp}] {self.agent_id} is now available again."
        self.log.append(entry)
        return entry

    def respond(self, emergency_type, location):
        """
        Core response method — overridden by each specific agent.
        Base version just returns a generic acknowledgement.
        """
        return {
            "agent_id"       : self.agent_id,
            "agent_type"     : self.agent_type,
            "emergency_type" : emergency_type,
            "destination"    : location,
            "status"         : "dispatched",
            "route"          : "PENDING — Basil's routing engine will fill this in Sprint 2"
        }

    def get_log(self):
        """Print the full activity log for this agent."""
        print(f"\n── Activity Log: {self.agent_id} ──")
        if not self.log:
            print("  No activity yet.")
        for entry in self.log:
            print(f"  {entry}")

    def __repr__(self):
        return f"<Agent {self.agent_id} | type={self.agent_type} | status={self.status}>"


# ── Quick test of the base class ───────────────
test_agent = Agent("TEST_01", "ambulance")

print("Agent created:  ", test_agent)
print("Is available?   ", test_agent.is_available())
print("Assigning...    ", test_agent.assign("cardiac arrest at Main St"))
print("Status now:     ", test_agent.status)
print("Completing...   ", test_agent.complete())
print("Status now:     ", test_agent.status)
test_agent.get_log()

Agent created:   <Agent TEST_01 | type=ambulance | status=available>
Is available?    True
Assigning...     [19:05:13] TEST_01 assigned to: cardiac arrest at Main St
Status now:      busy
Completing...    [19:05:13] TEST_01 is now available again.
Status now:      available

── Activity Log: TEST_01 ──
  [19:05:13] TEST_01 assigned to: cardiac arrest at Main St
  [19:05:13] TEST_01 is now available again.


---
## Cell 3 - Ambulance Agent

The `AmbulanceAgent` inherits from the base `Agent` class and adds 
medical-specific logic on top.

Its job is to:
- Check if it is available
- Assess the severity of the emergency
- Decide how many crew members are needed
- Return a structured dispatch response

> **Dummy data note:** Hospital locations and real unit positions are 
> placeholders for now. Diyona's dataset will replace these in Sprint 2.

In [7]:
# ─────────────────────────────────────────────
# CELL 3 — Ambulance Agent
# ─────────────────────────────────────────────

class AmbulanceAgent(Agent):
    """
    Handles medical emergencies.
    Inherits all base behaviour from Agent and adds medical-specific logic.
    """

    # Dummy hospital data — will be replaced by Diyona's dataset in Sprint 2
    HOSPITALS = {
        "City General"   : "LOCATION_A",
        "St. Mary's"     : "LOCATION_B",
        "North Medical"  : "LOCATION_C",
    }

    # Which emergencies need advanced life support (ALS) vs basic life support (BLS)
    ALS_REQUIRED = ["cardiac_arrest", "car_accident_major", "building_collapse", "drowning"]

    def __init__(self, agent_id, location="DEPOT_AMBULANCE"):
        super().__init__(agent_id, agent_type="ambulance", location=location)
        self.crew_size   = 2      # default crew
        self.nearest_hospital = "City General"   # placeholder

    def assess_severity(self, emergency_type):
        """
        Returns care level needed based on emergency type.
        ALS = Advanced Life Support (critical)
        BLS = Basic Life Support (standard)
        """
        if emergency_type in self.ALS_REQUIRED:
            return "ALS"
        return "BLS"

    def get_crew_size(self, care_level):
        """Critical emergencies need a bigger crew."""
        return 3 if care_level == "ALS" else 2

    def respond(self, emergency_type, location):
        """
        Override base respond() with ambulance-specific logic.
        Returns a full dispatch response dictionary.
        """
        if not self.is_available():
            return {
                "agent_id" : self.agent_id,
                "status"   : "unavailable",
                "message"  : f"{self.agent_id} is busy — another unit must respond."
            }

        # Assess and prepare
        care_level = self.assess_severity(emergency_type)
        crew       = self.get_crew_size(care_level)

        # Assign the agent (marks as busy, logs it)
        self.assign(f"{emergency_type} at {location}")

        return {
            "agent_id"         : self.agent_id,
            "agent_type"       : "ambulance",
            "emergency_type"   : emergency_type,
            "destination"      : location,
            "care_level"       : care_level,
            "crew_size"        : crew,
            "nearest_hospital" : self.nearest_hospital,
            "status"           : "dispatched",
            "route"            : "PENDING — Basil's routing engine (Sprint 2)"
        }


# ── Test the Ambulance Agent ───────────────────

amb1 = AmbulanceAgent("AMB_01")
amb2 = AmbulanceAgent("AMB_02")

# Test 1 — critical emergency
print("=" * 55)
print("TEST 1 — Cardiac Arrest (critical)")
print("=" * 55)
response1 = amb1.respond("cardiac_arrest", "Main Street")
for key, value in response1.items():
    print(f"  {key:<20} : {value}")

# Test 2 — standard emergency
print("\n" + "=" * 55)
print("TEST 2 — Minor Car Accident (standard)")
print("=" * 55)
response2 = amb2.respond("car_accident_minor", "Park Avenue")
for key, value in response2.items():
    print(f"  {key:<20} : {value}")

# Test 3 — what happens when agent is busy
print("\n" + "=" * 55)
print("TEST 3 — AMB_01 is busy, another call comes in")
print("=" * 55)
response3 = amb1.respond("drowning", "River Road")
for key, value in response3.items():
    print(f"  {key:<20} : {value}")

# Activity log
amb1.get_log()

TEST 1 — Cardiac Arrest (critical)
  agent_id             : AMB_01
  agent_type           : ambulance
  emergency_type       : cardiac_arrest
  destination          : Main Street
  care_level           : ALS
  crew_size            : 3
  nearest_hospital     : City General
  status               : dispatched
  route                : PENDING — Basil's routing engine (Sprint 2)

TEST 2 — Minor Car Accident (standard)
  agent_id             : AMB_02
  agent_type           : ambulance
  emergency_type       : car_accident_minor
  destination          : Park Avenue
  care_level           : BLS
  crew_size            : 2
  nearest_hospital     : City General
  status               : dispatched
  route                : PENDING — Basil's routing engine (Sprint 2)

TEST 3 — AMB_01 is busy, another call comes in
  agent_id             : AMB_01
  status               : unavailable
  message              : AMB_01 is busy — another unit must respond.

── Activity Log: AMB_01 ──
  [19:05:13] AMB_01 a

---
## Cell 4 - Fire Agent

The `FireAgent` handles fire and hazardous material emergencies.
It adds fire-specific logic on top of the base class:

- Assesses whether the fire is structural, vehicular, or a hazmat situation
- Decides what equipment is needed based on fire type
- Tracks how many units (trucks) to dispatch
- Returns a structured dispatch response consistent with the other agents

> **Dummy data note:** Fire station locations are placeholders —  
> Diyona's dataset replaces these in Sprint 2.

In [9]:
# ─────────────────────────────────────────────
# CELL 4 — Fire Agent
# ─────────────────────────────────────────────

class FireAgent(Agent):
    """
    Handles fire and hazardous material emergencies.
    Inherits all base behaviour from Agent and adds fire-specific logic.
    """

    # Dummy fire station data — replaced by Diyona's dataset in Sprint 2
    FIRE_STATIONS = {
        "Central Station" : "LOCATION_F1",
        "North Station"   : "LOCATION_F2",
        "East Station"    : "LOCATION_F3",
    }

    # Equipment needed per emergency type
    EQUIPMENT_MAP = {
        "fire"              : ["hose_lines", "breathing_apparatus", "ladder_truck"],
        "car_accident_major": ["hydraulic_rescue_tools", "hose_lines"],
        "building_collapse" : ["search_rescue_kit", "breathing_apparatus", "hose_lines"],
        "gas_leak"          : ["hazmat_suit", "gas_detector", "breathing_apparatus"],
    }

    # How many trucks to send per emergency type
    TRUCKS_NEEDED = {
        "fire"              : 2,
        "car_accident_major": 1,
        "building_collapse" : 3,
        "gas_leak"          : 1,
    }

    def __init__(self, agent_id, location="DEPOT_FIRE"):
        super().__init__(agent_id, agent_type="fire", location=location)
        self.nearest_station = "Central Station"    # placeholder

    def get_equipment(self, emergency_type):
        """Returns list of equipment needed for this emergency type."""
        return self.EQUIPMENT_MAP.get(emergency_type, ["standard_kit"])

    def get_trucks_needed(self, emergency_type):
        """Returns number of trucks to dispatch."""
        return self.TRUCKS_NEEDED.get(emergency_type, 1)

    def is_hazmat(self, emergency_type):
        """Flags whether this is a hazardous materials situation."""
        return emergency_type == "gas_leak"

    def respond(self, emergency_type, location):
        """
        Override base respond() with fire-specific logic.
        Returns a full dispatch response dictionary.
        """
        if not self.is_available():
            return {
                "agent_id" : self.agent_id,
                "status"   : "unavailable",
                "message"  : f"{self.agent_id} is busy — another unit must respond."
            }

        # Prepare response details
        equipment     = self.get_equipment(emergency_type)
        trucks        = self.get_trucks_needed(emergency_type)
        hazmat        = self.is_hazmat(emergency_type)

        # Assign the agent (marks as busy, logs it)
        self.assign(f"{emergency_type} at {location}")

        return {
            "agent_id"        : self.agent_id,
            "agent_type"      : "fire",
            "emergency_type"  : emergency_type,
            "destination"     : location,
            "trucks_dispatch" : trucks,
            "equipment"       : equipment,
            "hazmat_situation": hazmat,
            "nearest_station" : self.nearest_station,
            "status"          : "dispatched",
            "route"           : "PENDING — Basil's routing engine (Sprint 2)"
        }


# ── Test the Fire Agent ────────────────────────

fire1 = FireAgent("FIRE_01")
fire2 = FireAgent("FIRE_02")

# Test 1 — structural fire
print("=" * 55)
print("TEST 1 — Structural Fire")
print("=" * 55)
response1 = fire1.respond("fire", "Oak Street Apartments")
for key, value in response1.items():
    print(f"  {key:<20} : {value}")

# Test 2 — gas leak (hazmat)
print("\n" + "=" * 55)
print("TEST 2 — Gas Leak (hazmat situation)")
print("=" * 55)
response2 = fire2.respond("gas_leak", "Industrial Road")
for key, value in response2.items():
    print(f"  {key:<20} : {value}")

# Test 3 — building collapse (maximum response)
print("\n" + "=" * 55)
print("TEST 3 — Building Collapse (maximum response)")
print("=" * 55)
fire3 = FireAgent("FIRE_03")
response3 = fire3.respond("building_collapse", "Downtown Plaza")
for key, value in response3.items():
    print(f"  {key:<20} : {value}")

# Test 4 — FIRE_01 is busy
print("\n" + "=" * 55)
print("TEST 4 — FIRE_01 already busy")
print("=" * 55)
response4 = fire1.respond("fire", "Hill Road")
for key, value in response4.items():
    print(f"  {key:<20} : {value}")

fire1.get_log()

TEST 1 — Structural Fire
  agent_id             : FIRE_01
  agent_type           : fire
  emergency_type       : fire
  destination          : Oak Street Apartments
  trucks_dispatch      : 2
  equipment            : ['hose_lines', 'breathing_apparatus', 'ladder_truck']
  hazmat_situation     : False
  nearest_station      : Central Station
  status               : dispatched
  route                : PENDING — Basil's routing engine (Sprint 2)

TEST 2 — Gas Leak (hazmat situation)
  agent_id             : FIRE_02
  agent_type           : fire
  emergency_type       : gas_leak
  destination          : Industrial Road
  trucks_dispatch      : 1
  equipment            : ['hazmat_suit', 'gas_detector', 'breathing_apparatus']
  hazmat_situation     : True
  nearest_station      : Central Station
  status               : dispatched
  route                : PENDING — Basil's routing engine (Sprint 2)

TEST 3 — Building Collapse (maximum response)
  agent_id             : FIRE_03
  agent_type 

---
## Cell 5 - Police Agent

The `PoliceAgent` handles crime, crowd control, and scene security.
It is also the **default responder** for unknown emergencies — it gets  
sent first to assess and call in other services if needed.

Police-specific logic includes:
- Assessing whether armed response is needed
- Deciding how many units (cars) to dispatch
- Flagging whether the scene needs to be cordoned off
- Acting as lead coordinator when multiple agents respond together

In [11]:
# ─────────────────────────────────────────────
# CELL 5 — Police Agent
# ─────────────────────────────────────────────

class PoliceAgent(Agent):
    """
    Handles crime, crowd control, and scene security.
    Also the default first responder for unknown emergencies.
    Inherits all base behaviour from Agent and adds police-specific logic.
    """

    # Dummy station data — replaced by Diyona's dataset in Sprint 2
    POLICE_STATIONS = {
        "Central HQ"    : "LOCATION_P1",
        "North Precinct": "LOCATION_P2",
        "East Precinct" : "LOCATION_P3",
    }

    # Whether armed response unit is needed
    ARMED_RESPONSE_REQUIRED = ["robbery", "assault", "building_collapse", "unknown"]

    # Units (cars) to dispatch per emergency type
    UNITS_NEEDED = {
        "robbery"           : 2,
        "assault"           : 2,
        "car_accident_minor": 1,
        "car_accident_major": 2,
        "building_collapse" : 3,
        "gas_leak"          : 2,
        "drowning"          : 1,
        "unknown"           : 1,
    }

    # Whether the scene needs to be cordoned off
    CORDON_REQUIRED = ["car_accident_major", "building_collapse", "gas_leak", "fire"]

    def __init__(self, agent_id, location="DEPOT_POLICE"):
        super().__init__(agent_id, agent_type="police", location=location)
        self.nearest_station = "Central HQ"     # placeholder
        self.is_lead         = False            # True when coordinating multi-agent response

    def needs_armed_response(self, emergency_type):
        """Returns True if armed response unit should be dispatched."""
        return emergency_type in self.ARMED_RESPONSE_REQUIRED

    def get_units_needed(self, emergency_type):
        """Returns number of police cars to dispatch."""
        return self.UNITS_NEEDED.get(emergency_type, 1)

    def needs_cordon(self, emergency_type):
        """Returns True if police need to cordon off the scene."""
        return emergency_type in self.CORDON_REQUIRED

    def set_as_lead(self):
        """
        Sets this police unit as the lead coordinator.
        Used when multiple agents respond to the same incident.
        """
        self.is_lead = True

    def respond(self, emergency_type, location):
        """
        Override base respond() with police-specific logic.
        Returns a full dispatch response dictionary.
        """
        if not self.is_available():
            return {
                "agent_id" : self.agent_id,
                "status"   : "unavailable",
                "message"  : f"{self.agent_id} is busy — another unit must respond."
            }

        # Prepare response details
        armed_response = self.needs_armed_response(emergency_type)
        units          = self.get_units_needed(emergency_type)
        cordon         = self.needs_cordon(emergency_type)

        # Assign the agent (marks as busy, logs it)
        self.assign(f"{emergency_type} at {location}")

        return {
            "agent_id"        : self.agent_id,
            "agent_type"      : "police",
            "emergency_type"  : emergency_type,
            "destination"     : location,
            "units_dispatch"  : units,
            "armed_response"  : armed_response,
            "cordon_required" : cordon,
            "lead_coordinator": self.is_lead,
            "nearest_station" : self.nearest_station,
            "status"          : "dispatched",
            "route"           : "PENDING — Basil's routing engine (Sprint 2)"
        }


# ── Test the Police Agent ──────────────────────

pol1 = PoliceAgent("POL_01")
pol2 = PoliceAgent("POL_02")

# Test 1 — robbery (armed response needed)
print("=" * 55)
print("TEST 1 — Robbery (armed response)")
print("=" * 55)
response1 = pol1.respond("robbery", "Bank Street")
for key, value in response1.items():
    print(f"  {key:<20} : {value}")

# Test 2 — car accident (scene security, no arms)
print("\n" + "=" * 55)
print("TEST 2 — Major Car Accident (scene cordon)")
print("=" * 55)
response2 = pol2.respond("car_accident_major", "Highway 5")
for key, value in response2.items():
    print(f"  {key:<20} : {value}")

# Test 3 — unknown emergency (default first responder)
print("\n" + "=" * 55)
print("TEST 3 — Unknown Emergency (default responder)")
print("=" * 55)
pol3 = PoliceAgent("POL_03")
pol3.set_as_lead()
response3 = pol3.respond("unknown", "Riverside Drive")
for key, value in response3.items():
    print(f"  {key:<20} : {value}")

# Test 4 — POL_01 already busy
print("\n" + "=" * 55)
print("TEST 4 — POL_01 already busy")
print("=" * 55)
response4 = pol1.respond("assault", "Market Road")
for key, value in response4.items():
    print(f"  {key:<20} : {value}")

pol1.get_log()

TEST 1 — Robbery (armed response)
  agent_id             : POL_01
  agent_type           : police
  emergency_type       : robbery
  destination          : Bank Street
  units_dispatch       : 2
  armed_response       : True
  cordon_required      : False
  lead_coordinator     : False
  nearest_station      : Central HQ
  status               : dispatched
  route                : PENDING — Basil's routing engine (Sprint 2)

TEST 2 — Major Car Accident (scene cordon)
  agent_id             : POL_02
  agent_type           : police
  emergency_type       : car_accident_major
  destination          : Highway 5
  units_dispatch       : 2
  armed_response       : False
  cordon_required      : True
  lead_coordinator     : False
  nearest_station      : Central HQ
  status               : dispatched
  route                : PENDING — Basil's routing engine (Sprint 2)

TEST 3 — Unknown Emergency (default responder)
  agent_id             : POL_03
  agent_type           : police
  emergency_t

---
## Cell 6 - Dispatch Function

This is the core of the entire multi-agent system.

The `dispatch()` function:
1. Receives an emergency type and location
2. Looks up `DISPATCH_TABLE` from Cell 1 to find which agents respond
3. Creates the right agent instances
4. Calls `respond()` on each one
5. If multiple agents respond, sets police as lead coordinator
6. Returns a single combined **Incident Report**

This is what Vyshnavi's integration layer will call in Sprint 2 —  
she passes in the LLM-extracted emergency type and location,  
and this function handles everything from there.

In [13]:
# ─────────────────────────────────────────────
# CELL 6 — Dispatch Function
# ─────────────────────────────────────────────

from datetime import datetime

# ── Agent pool ────────────────────────────────
AGENT_POOL = {
    "ambulance": [AmbulanceAgent("AMB_01"), AmbulanceAgent("AMB_02")],
    "fire"     : [FireAgent("FIRE_01"),     FireAgent("FIRE_02")],
    "police"   : [PoliceAgent("POL_01"),    PoliceAgent("POL_02")],
}

def reset_agent_pool():
    """
    Resets all agents to available and clears their logs.
    Called before each test to simulate a fresh independent incident.
    """
    for agent_type, agents in AGENT_POOL.items():
        for agent in agents:
            agent.status = "available"
            agent.log    = []
            if hasattr(agent, "is_lead"):
                agent.is_lead = False

def get_available_agent(agent_type):
    """
    Finds the first available agent of the given type from the pool.
    Returns None if all units of that type are busy.
    """
    for agent in AGENT_POOL[agent_type]:
        if agent.is_available():
            return agent
    return None

def dispatch(emergency_type, location):
    """
    Main dispatch function — the entry point for the whole system.

    Parameters:
        emergency_type (str) : must match a key in DISPATCH_TABLE
        location       (str) : plain text location for now (coords in Sprint 2)

    Returns:
        dict : full incident report with all agent responses
    """
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    # ── Step 1: Look up dispatch table ────────
    if emergency_type not in DISPATCH_TABLE:
        emergency_type = "unknown"

    dispatch_info = DISPATCH_TABLE[emergency_type]
    agents_needed = dispatch_info["agents"]
    priority      = dispatch_info["priority"]
    notes         = dispatch_info["notes"]

    # ── Step 2: Collect responses ─────────────
    responses   = []
    agents_sent = []
    unavailable = []

    for agent_type in agents_needed:
        agent = get_available_agent(agent_type)

        if agent is None:
            unavailable.append(agent_type)
            continue

        # If multiple agents respond, set police as lead coordinator
        if agent_type == "police" and len(agents_needed) > 1:
            agent.set_as_lead()

        response = agent.respond(emergency_type, location)
        responses.append(response)
        agents_sent.append(agent.agent_id)

    # ── Step 3: Build incident report ─────────
    incident_report = {
        "incident_id"    : f"INC_{datetime.now().strftime('%H%M%S')}",
        "timestamp"      : timestamp,
        "emergency_type" : emergency_type,
        "location"       : location,
        "priority"       : priority,
        "dispatch_notes" : notes,
        "agents_sent"    : agents_sent,
        "unavailable"    : unavailable if unavailable else "none",
        "total_responses": len(responses),
        "responses"      : responses,
    }

    return incident_report

def print_incident_report(report):
    """Pretty prints the full incident report."""
    print("\n" + "=" * 60)
    print(f"  INCIDENT REPORT — {report['incident_id']}")
    print("=" * 60)
    print(f"  Timestamp      : {report['timestamp']}")
    print(f"  Emergency      : {report['emergency_type'].upper()}")
    print(f"  Location       : {report['location']}")
    print(f"  Priority       : {report['priority']} (1=critical)")
    print(f"  Notes          : {report['dispatch_notes']}")
    print(f"  Agents sent    : {', '.join(report['agents_sent'])}")
    print(f"  Unavailable    : {report['unavailable']}")
    print(f"  Total responses: {report['total_responses']}")
    print("\n  ── Agent Responses ──")
    for r in report["responses"]:
        print(f"\n  [{r['agent_type'].upper()}] {r['agent_id']}")
        for key, value in r.items():
            if key not in ["agent_id", "agent_type"]:
                print(f"    {key:<20} : {value}")
    print("=" * 60)


# ── Test the Dispatch Function ─────────────────

# Test 1 — single agent (robbery)
reset_agent_pool()
print_incident_report(dispatch("robbery", "Bank Street"))

# Test 2 — two agents (assault)
reset_agent_pool()
print_incident_report(dispatch("assault", "Market Road"))

# Test 3 — all three agents (building collapse)
reset_agent_pool()
print_incident_report(dispatch("building_collapse", "Downtown Plaza"))

# Test 4 — unknown emergency
reset_agent_pool()
print_incident_report(dispatch("unknown", "Riverside Drive"))


  INCIDENT REPORT — INC_190513
  Timestamp      : 2026-03-20 19:05:13
  Emergency      : ROBBERY
  Location       : Bank Street
  Priority       : 2 (1=critical)
  Notes          : Police only
  Agents sent    : POL_01
  Unavailable    : none
  Total responses: 1

  ── Agent Responses ──

  [POLICE] POL_01
    emergency_type       : robbery
    destination          : Bank Street
    units_dispatch       : 2
    armed_response       : True
    cordon_required      : False
    lead_coordinator     : False
    nearest_station      : Central HQ
    status               : dispatched
    route                : PENDING — Basil's routing engine (Sprint 2)

  INCIDENT REPORT — INC_190513
  Timestamp      : 2026-03-20 19:05:13
  Emergency      : ASSAULT
  Location       : Market Road
  Priority       : 2 (1=critical)
  Notes          : Police to secure scene, ambulance for victim
  Agents sent    : POL_01, AMB_01
  Unavailable    : none
  Total responses: 2

  ── Agent Responses ──

  [POLICE] 

---
## Cell 7 - Full System Test (Sprint 1 Demo)

This is the final test — it runs all 10 emergency types from  
`DISPATCH_TABLE` through the dispatch function one by one.

Before each test the agent pool is reset so every emergency  
starts with all units available - simulating independent incidents.

This cell is **Sprint 1 demo output** to show the team.

In [15]:
# ─────────────────────────────────────────────
# CELL 7 — Full System Test (Sprint 1 Demo)
# ─────────────────────────────────────────────

def run_full_system_test():
    """
    Runs all 10 emergency types through dispatch and prints a summary.
    This is the Sprint 1 demo — shows the full system working end to end.
    """

    test_scenarios = [
        ("cardiac_arrest",      "Main Street"),
        ("fire",                "Oak Street Apartments"),
        ("car_accident_minor",  "Park Avenue"),
        ("car_accident_major",  "Highway 5"),
        ("robbery",             "Bank Street"),
        ("assault",             "Market Road"),
        ("building_collapse",   "Downtown Plaza"),
        ("gas_leak",            "Industrial Road"),
        ("drowning",            "River Road"),
        ("unknown",             "Riverside Drive"),
    ]

    # ── Summary table ──────────────────────────
    print("\n" + "=" * 75)
    print("  SPRINT 1 DEMO — MULTI-AGENT DISPATCH SYSTEM")
    print("  All 10 emergency types — independent incidents")
    print("=" * 75)
    print(f"  {'EMERGENCY':<25} {'PRIORITY':<10} {'AGENTS DISPATCHED':<30} {'STATUS'}")
    print("  " + "─" * 71)

    summary = []

    for emergency_type, location in test_scenarios:
        reset_agent_pool()
        report     = dispatch(emergency_type, location)
        agents_str = ", ".join(report["agents_sent"]) if report["agents_sent"] else "NONE AVAILABLE"
        status     = "OK" if report["total_responses"] > 0 else "FAILED"
        summary.append((emergency_type, report["priority"], agents_str, status))
        print(f"  {emergency_type:<25} {report['priority']:<10} {agents_str:<30} {status}")

    # ── Final stats ────────────────────────────
    total    = len(summary)
    ok       = sum(1 for s in summary if s[3] == "OK")
    failed   = total - ok
    critical = sum(1 for s in summary if s[1] == 1)

    print("  " + "─" * 71)
    print(f"\n  Total scenarios         : {total}")
    print(f"  Successfully dispatched : {ok}/{total}")
    print(f"  Failed dispatches       : {failed}")
    print(f"  Critical (P1) incidents : {critical}")
    print("\n  All routes are PENDING — Basil's routing engine connects in Sprint 2")
    print("=" * 75)

    # ── Detailed reports for critical incidents only ───
    print("\n\n  DETAILED REPORTS — Priority 1 (Critical) Incidents Only")

    critical_scenarios = [(e, l) for e, l in test_scenarios
                          if DISPATCH_TABLE[e]["priority"] == 1]

    for emergency_type, location in critical_scenarios:
        reset_agent_pool()
        report = dispatch(emergency_type, location)
        print_incident_report(report)


# ── Run it ─────────────────────────────────────
run_full_system_test()


  SPRINT 1 DEMO — MULTI-AGENT DISPATCH SYSTEM
  All 10 emergency types — independent incidents
  EMERGENCY                 PRIORITY   AGENTS DISPATCHED              STATUS
  ───────────────────────────────────────────────────────────────────────
  cardiac_arrest            1          AMB_01                         OK
  fire                      1          FIRE_01, AMB_01                OK
  car_accident_minor        2          AMB_01, POL_01                 OK
  car_accident_major        1          AMB_01, FIRE_01, POL_01        OK
  robbery                   2          POL_01                         OK
  assault                   2          POL_01, AMB_01                 OK
  building_collapse         1          FIRE_01, AMB_01, POL_01        OK
  gas_leak                  1          FIRE_01, POL_01                OK
  drowning                  1          AMB_01, POL_01                 OK
  unknown                   3          POL_01                         OK
  ─────────────────────

---

## Sprint 1 Complete ✓

**Manya Mahajan — AI Agent Developer**  
**Date completed:** Week 3, Sprint 1

---

### What was built in this notebook

| Cell | Component | Status |
|---|---|---|
| Cell 1 | Dispatch table - 10 emergency types | Done |
| Cell 2 | Base `Agent` class | Done |
| Cell 3 | `AmbulanceAgent` - ALS/BLS, crew sizing | Done |
| Cell 4 | `FireAgent` - equipment, hazmat, trucks | Done |
| Cell 5 | `PoliceAgent` - armed response, cordon, lead | Done |
| Cell 6 | `dispatch()` - full incident report generator | Done |
| Cell 7 | Full system test - 10/10 scenarios passing | Done |

---

### Sprint 1 results

| Metric | Result |
|---|---|
| Total scenarios run | 10/10 |
| Successfully dispatched | 10/10 |
| Failed dispatches | 0 |
| Critical (P1) incidents handled | 6/6 |
| Units showing unavailable | 0 |
| Lead coordinator assigned correctly | Yes — all multi-agent responses |

---

### What is placeholder — replaced in Sprint 2

| Placeholder | Who replaces it | When |
|---|---|---|
| `route: PENDING` | Basil - routing engine | Sprint 2, Week 4 |
| Hardcoded locations | Diyona - real GIS datasets | Sprint 2, Week 4 |
| Manual emergency type input | Vyshnavi - LLM parsing layer | Sprint 2, Week 5 |
| 2 units per agent type | Expanded live unit pool | Sprint 2, Week 5 |


---

> *Completed independently in Week 3, Sprint 1.  
> No external dependencies beyond Python's standard `datetime` library.  
> All location and unit data is placeholder — replacing with real datasets in Sprint 2.*